In [5]:
import os
import numpy as np
import pandas as pd
from scipy.stats import pearsonr, spearmanr

CLASSICAL_PATH = "all_csv_summary"
CODEBLEU_PATH = "codebleu_results_group2_full.csv"

K = 10

rows = []
global_struct = []
global_perf = []

# ============================================================
# Classical Methods
# ============================================================

for file in os.listdir(CLASSICAL_PATH):

    if not file.endswith(".csv"):
        continue

    if file == "summary_scatter_sampled.csv":
        continue

    df = pd.read_csv(os.path.join(CLASSICAL_PATH, file))

    struct = df["distance_struct"].values
    perf = df["distance_perf"].values

    task = file.split("_")[0] + "_" + file.split("_")[1]
    method = file.split("_")[2]

    r_p, p_p = pearsonr(struct, perf)
    r_s, p_s = spearmanr(struct, perf)

    # Overlap@10
    top_struct = np.argsort(struct)[:K]
    top_perf = np.argsort(perf)[:K]
    overlap = len(set(top_struct) & set(top_perf)) / K

    rows.append([
        task, method,
        r_p, p_p,
        r_s, p_s,
        overlap
    ])

    global_struct.extend(struct)
    global_perf.extend(perf)

# ============================================================
# CodeBLEU
# ============================================================

cb_df = pd.read_csv(CODEBLEU_PATH)

for task, group in cb_df.groupby("raw_app_type"):

    struct = group["d_codebleu"].values
    perf = group["cb_d_perf"].values

    r_p, p_p = pearsonr(struct, perf)
    r_s, p_s = spearmanr(struct, perf)

    top_struct = np.argsort(struct)[:K]
    top_perf = np.argsort(perf)[:K]
    overlap = len(set(top_struct) & set(top_perf)) / K

    rows.append([
        task, "codebleu",
        r_p, p_p,
        r_s, p_s,
        overlap
    ])

    global_struct.extend(struct)
    global_perf.extend(perf)

# ============================================================
# Global
# ============================================================

global_struct = np.array(global_struct)
global_perf = np.array(global_perf)

g_r_p, g_p_p = pearsonr(global_struct, global_perf)
g_r_s, g_p_s = spearmanr(global_struct, global_perf)

# ============================================================
# Save
# ============================================================

results_df = pd.DataFrame(rows, columns=[
    "task", "method",
    "pearson_r", "pearson_p",
    "spearman_r", "spearman_p",
    "overlap@10"
])

results_df.to_csv("FAST_statistical_summary.csv", index=False)

print("\n==== Per Task Results ====\n")
print(results_df)

print("\n==== Global ====\n")
print(f"Pearson:  r={g_r_p:.4f}, p={g_p_p:.4e}")
print(f"Spearman: r={g_r_s:.4f}, p={g_p_s:.4e}")



==== Per Task Results ====

                    task    method  pearson_r      pearson_p  spearman_r  \
0             bin_greedy       ast   0.036330   2.767390e-07    0.004559   
1             bin_greedy   jaccard   0.031209   1.014920e-05   -0.009699   
2             bin_greedy     tfidf   0.096584   1.185072e-42    0.095192   
3               cvrp_lns       ast   0.087993   1.172285e-35    0.101932   
4               cvrp_lns   jaccard   0.054693   1.010300e-14    0.038806   
5               cvrp_lns     tfidf   0.011285   1.106262e-01   -0.010287   
6   premarshalling_astar       ast   0.005303   4.533377e-01    0.025703   
7   premarshalling_astar   jaccard   0.058306   1.559822e-16    0.036797   
8   premarshalling_astar     tfidf   0.044137   4.258370e-10    0.038898   
9           puzzle_astar       ast   0.038859   3.868727e-08    0.039905   
10          puzzle_astar   jaccard   0.075743   7.740489e-27    0.075830   
11          puzzle_astar     tfidf  -0.060268   1.469271e-1